In [19]:
from pathlib import Path
import os

import pandas as pd
from caf.base import DVector, ZoningSystem

os.chdir(r'..\..')
from land_use.data_processing import create_dvector_from_data, read_dvector_data
os.chdir(r'supporting_processes\norcom')

In [2]:
# read in the 2021 census population
input_dir = Path(r"F:\Deliverables\Land-Use\241213_Population\01_Intermediate Files")
input_files = input_dir.glob("Output P10_*.hdf")
dvec_dict = {}

In [3]:
for file in input_files:
    print(file)
    _2021_data = DVector.load(file).add_segments(['total']).aggregate(['total'])
    dvec_dict[file.with_suffix("").name.split("_")[1]] = _2021_data

F:\Deliverables\Land-Use\241213_Population\01_Intermediate Files\Output P10_EM.hdf
F:\Deliverables\Land-Use\241213_Population\01_Intermediate Files\Output P10_EoE.hdf
F:\Deliverables\Land-Use\241213_Population\01_Intermediate Files\Output P10_Lon.hdf
F:\Deliverables\Land-Use\241213_Population\01_Intermediate Files\Output P10_NE.hdf
F:\Deliverables\Land-Use\241213_Population\01_Intermediate Files\Output P10_NW.hdf
F:\Deliverables\Land-Use\241213_Population\01_Intermediate Files\Output P10_SE.hdf
F:\Deliverables\Land-Use\241213_Population\01_Intermediate Files\Output P10_SW.hdf
F:\Deliverables\Land-Use\241213_Population\01_Intermediate Files\Output P10_Wales.hdf
F:\Deliverables\Land-Use\241213_Population\01_Intermediate Files\Output P10_WM.hdf
F:\Deliverables\Land-Use\241213_Population\01_Intermediate Files\Output P10_YH.hdf


In [15]:
# read in LSOA CSV and create DVector of zone areas
zone_areas = pd.read_csv(r"F:\Working\Land-Use\SHAPEFILES\LSOA (2021)\LSOA_2021_EnglandWales.csv")
zone_areas["total"] = 1
zone_areas.head()

,LSOA21CD,LSOA21NM,X,Y,MSOA21CD,LAD21CD,MSOA21NM,LAD21NM,RGN21CD,RGN21NM,area_m2,total
0,E01000001,City of London 001A,532150.839556,181617.460935,E02000001,E09000001,City of London 001,City of London,E12000007,London,129913.1890,1
1,E01000002,City of London 001B,532443.368490,181643.603261,E02000001,E09000001,City of London 001,City of London,E12000007,London,228503.5543,1
2,E01000003,City of London 001C,532205.545275,182030.048913,E02000001,E09000001,City of London 001,City of London,E12000007,London,59075.9538,1
3,E01000005,City of London 001E,533620.049356,181152.840732,E02000001,E09000001,City of London 001,City of London,E12000007,London,189645.7776,1
4,E01000006,Barking and Dagenham 016A,544933.450792,184296.480909,E02000017,E09000002,Barking and Dagenham 016,Barking and Dagenham,E12000007,London,146578.2629,1


In [16]:
# pivot to dvector format
pivoted = pd.pivot_table(data=zone_areas, values='area_m2', index=['total'], columns=['LSOA21CD'])
pivoted

LSOA21CD,E01000001,E01000002,E01000003,E01000005,E01000006,E01000007,E01000008,E01000009,E01000011,E01000012,...,W01002031,W01002032,W01002033,W01002034,W01002035,W01002036,W01002037,W01002038,W01002039,W01002040
total,,,,,,,,,,,,,,,,,,,,,
1,129913.189,228503.5543,59075.9538,189645.7776,146578.2629,200151.4672,193702.8066,127996.2651,91658.44438,140469.128,...,243060.222,1764822.138,11147092.35,779249.5247,310830.7386,388943.2832,284531.1856,3497250.136,650904.8771,681595.779


In [17]:
# create dvector
area = create_dvector_from_data(pivoted, geographical_level="LSOA2021", input_segments=["total"])

C:\Miniforge3\envs\normits_lu\Lib\site-packages\caf\base\data_structures.py:443: UserWarning: 0.0 dropped on seg validation.
  warnings.warn(f"{full_sum - cut_sum} dropped on seg validation.")


In [18]:
# save to dvector and load in as region specific ones
area.save(r"I:\NorMITs NorCOM\AT Population Densities\LSOA_area_m2.hdf")

In [21]:
for GOR, pop_data in dvec_dict.items():
    gor_area = read_dvector_data(
        file_path=Path(r"I:\NorMITs NorCOM\AT Population Densities\LSOA_area_m2.hdf"),
        geographical_level="LSOA2021", input_segments=["total"], geography_subset=GOR, hdf_key="data"
    )
    densities = pop_data / gor_area
    densities.save(fr"I:\NorMITs NorCOM\AT Population Densities\LSOA_pop_per_m2_{GOR}.hdf")

The input data at I:\NorMITs NorCOM\AT Population Densities\LSOA_area_m2.hdf started with 35,672 columns. Filtering to EM results in 2,847 columns.
C:\Miniforge3\envs\normits_lu\Lib\site-packages\caf\base\data_structures.py:443: UserWarning: 0.0 dropped on seg validation.
  warnings.warn(f"{full_sum - cut_sum} dropped on seg validation.")
The input data at I:\NorMITs NorCOM\AT Population Densities\LSOA_area_m2.hdf started with 35,672 columns. Filtering to EoE results in 3,758 columns.
C:\Miniforge3\envs\normits_lu\Lib\site-packages\caf\base\data_structures.py:443: UserWarning: 0.0 dropped on seg validation.
  warnings.warn(f"{full_sum - cut_sum} dropped on seg validation.")
The input data at I:\NorMITs NorCOM\AT Population Densities\LSOA_area_m2.hdf started with 35,672 columns. Filtering to Lon results in 4,994 columns.
C:\Miniforge3\envs\normits_lu\Lib\site-packages\caf\base\data_structures.py:443: UserWarning: 0.0 dropped on seg validation.
  warnings.warn(f"{full_sum - cut_sum} drop